In [ ]:
import pandas as pd
import joblib
from geopy.distance import geodesic


In [ ]:
model = joblib.load("fixed_fraud_detection_model.jb")
encoder = joblib.load("fixed_label_encoder.jb")


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    return geodesic((lat1, lon1), (lat2, lon2)).km


In [ ]:
# Sample input dictionary
input_data = {
    'merchant': 'BestBuy',
    'category': 'Electronics',
    'amt': 250.00,
    'lat': 37.7749,
    'long': -122.4194,
    'merch_lat': 37.7750,
    'merch_long': -122.4195,
    'hour': 14,
    'day': 12,
    'month': 5,
    'gender': 'Male',
    'cc_num': '1234567890123456'
}


In [ ]:
distance = haversine(input_data['lat'], input_data['long'], input_data['merch_lat'], input_data['merch_long'])

df = pd.DataFrame([[
    input_data['merchant'], input_data['category'], input_data['amt'],
    distance, input_data['hour'], input_data['day'], input_data['month'],
    input_data['gender'], input_data['cc_num']
]], columns=['merchant', 'category', 'amt', 'distance', 'hour', 'day', 'month', 'gender', 'cc_num'])

# Encode categorical fields
categorical_col = ['merchant', 'category', 'gender']
for col in categorical_col:
    try:
        df[col] = encoder[col].transform(df[col])
    except ValueError:
        df[col] = -1

# Hash cc_num
df['cc_num'] = df['cc_num'].apply(lambda x: hash(x) % (10 ** 2))
df


In [ ]:
prediction = model.predict(df)[0]
result = "Fraudulent Transaction" if prediction == 1 else "Legitimate Transaction"
print("Prediction:", result)
